# Notebook 05: What did the generalized moment learn?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yangycpku/Machine_Learning_Macro_PSU/blob/main/Tutorials/Tutorial1/src/05_DeepHAM_Visualize.ipynb)

**Course:** Penn State Mini-Course on Machine Learning for Dynamic Economic Models (Penn State University, September 2026)
**Session:** Lecture 1 tutorial: Deep Learning for Solving Heterogeneous Agents Models (DeepHAM)
**Slides:** [`Lectures/`](https://github.com/yangycpku/Machine_Learning_Macro_PSU/tree/main/Lectures) in the course repository
**Notebook role:** core (in-class walkthrough)
**Author:** Yucheng Yang (University of Zurich and Swiss Finance Institute). [Course repository](https://github.com/yangycpku/Machine_Learning_Macro_PSU)

---

This notebook opens up a solved model and asks three questions.

1. **Does DeepHAM reproduce the Krusell–Smith aggregate dynamics?** Simulate both solutions
   under *identical* shocks and overlay the paths of $K_t$ and $C_t$.
2. **What is the learned basis function $\mathcal{Q}(a)$?** Reproduces Figure 3(a) of the paper.
3. **How does the value function depend on the generalized moment?** Reproduces Figure 3(b),
   and with it the redistribution result in Section 3.2.2.

By default it loads `game_nn_n50_1gm3`, the solved model shipped with this repository — the
run behind Figure 3 of the paper. Set `MODEL_DIR` to your own run from notebook 02 to look
at what you trained instead.


In [ ]:
RUN_MODE = "smoke"     # one of: "smoke", "teaching", "production"
SAVE_FIGURES = False   # write PDFs into ../output

## 1. Set up the code directory

DeepHAM is a package of plain Python modules (`param.py`, `dataset.py`, `value.py`,
`policy.py`, ...) that expect to be imported with `src/` as the working directory, with the
data alongside it in `../data`. The cell below handles both ways of running this notebook.

* **Google Colab** (the default for this course). The first run clones the course repository
  into `/content` (about 20 seconds) and moves into `Tutorials/Tutorial1/src`. Nothing needs
  to be installed: Colab already ships TensorFlow, NumPy, SciPy and matplotlib. A GPU is
  optional (`Runtime -> Change runtime type`); at the `smoke` and `teaching` budgets most of
  the wall clock is the NumPy simulation, so the free CPU runtime is fine.
* **A local clone.** Open the notebook from inside `Tutorials/Tutorial1/src` and the cell
  leaves the working directory alone.


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yangycpku/Machine_Learning_Macro_PSU.git"
REPO_DIR = "/content/Machine_Learning_Macro_PSU"
SRC_DIR = os.path.join(REPO_DIR, "Tutorials", "Tutorial1", "src")

try:
    import google.colab  # noqa: F401  (importable only on a Colab runtime)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isfile(os.path.join(SRC_DIR, "param.py")):
        print("Cloning the course repository into", REPO_DIR, "...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(SRC_DIR)

# Anywhere else (a local clone) the notebook's own folder is already src/.
if not os.path.isfile("param.py"):
    raise FileNotFoundError(
        f"Expected to be inside Tutorials/Tutorial1/src, but the working directory is "
        f"{os.getcwd()!r}. Open this notebook from inside src/, or os.chdir() there."
    )

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Running on Colab:", IN_COLAB)
print("Working directory:", os.getcwd())


In [ ]:
import json
import time

import numpy as np
import matplotlib
from matplotlib import pyplot as plt

from param import KSParam
from dataset import KSInitDataSet
from value import ValueTrainer
from policy import KSPolicyTrainer
from simulation_KS import simul_shocks, simul_k
from util import print_elapsedtime
from util import set_random_seed

set_random_seed(996)
plt.rcParams.update({"font.size": 14})

## 1. Load a solved model

Two solved models ship with this repository:

| directory | representation | corresponds to |
|---|---|---|
| `game_nn_n50_1fm1` | 1 fixed moment (mean capital) | notebook 01 |
| `game_nn_n50_1gm3` | 1 learned generalized moment | notebook 02, and Figure 3 of the paper |

The generalized-moment sections below need `n_gm > 0`, so they require the second one (or
your own `n_gm = 1` run).

`KSPolicyTrainer(..., policy_path=...)` does more than load weights: it also calls
`init_ds.load_stats(...)`, which restores the input normalisation used during training.
That matters — the networks are fed $(a - \mu)/\sigma$, so they should be evaluated under
the same $\mu, \sigma$ they were trained with.

In [ ]:
# Which solved model to open.
#   "game_nn_n50_1gm3"          -- shipped reference run, behind Figure 3 of the paper
#   "game_nn_n50_1gm_teaching"  -- your own run from notebook 02, if you trained one
MODEL_DIR = "../data/simul_results/KS/game_nn_n50_1gm3"

# Length of the comparison simulation. The paper uses 2000 periods and discards the first
# half; shorter runs are fine for looking at the mechanics.
T_SIM = {"smoke": 400, "teaching": 1000, "production": 2000}[RUN_MODE]
T_BURN = {"smoke": 300, "teaching": 1000, "production": 6000}[RUN_MODE]

with open(os.path.join(MODEL_DIR, "config.json"), "r") as f:
    config = json.load(f)

config["simul_config"]["T"] = T_SIM
config["dataset_config"]["n_path"] = config["simul_config"]["n_path"]
config["dataset_config"]["t_burn"] = T_BURN
config["init_with_bchmk"] = True   # burn in from the KS benchmark policy

start_time = time.monotonic()

mparam = KSParam(config["n_agt"], config["beta"], config["mats_path"])
init_ds = KSInitDataSet(mparam, config)

vtrainers = [ValueTrainer(config) for _ in range(config["value_config"]["num_vnet"])]
for i, vtr in enumerate(vtrainers):
    vtr.load_model(os.path.join(MODEL_DIR, f"value{i}.weights.h5"))

ptrainer = KSPolicyTrainer(vtrainers, init_ds, os.path.join(MODEL_DIR, "policy.weights.h5"))

n_path = config["simul_config"]["n_path"]
n_agt = config["n_agt"]
n_gm = config["n_gm"]
print(
    f"Loaded {MODEL_DIR}\n"
    f"  n_fm = {config['n_fm']}, n_gm = {n_gm}, "
    f"{config['value_config']['num_vnet']} value nets, {n_agt} agents"
)
print_elapsedtime(time.monotonic() - start_time)

## 2. The same shocks, two solutions

The comparison is only meaningful if both methods face an identical shock history, so the
aggregate and idiosyncratic shocks are drawn once and passed to both simulations. The
benchmark is the Krusell–Smith solution itself, stored as b-splines in
`../data/KS_policy_value_NS.mat` and evaluated with `policy_type="pde"`.

In [ ]:
start_time = time.monotonic()

state_init = init_ds.next_batch(n_path)
shocks = simul_shocks(n_path, T_SIM, mparam, state_init)
ashock, ishock = shocks[0], shocks[1]      # (paths, T) and (paths, n_agt, T)

simul_data_bchmk = simul_k(
    n_path, T_SIM, mparam, init_ds.k_policy_bchmk,
    policy_type="pde", state_init=state_init, shocks=shocks,
)
simul_data_nn = simul_k(
    n_path, T_SIM, mparam, ptrainer.current_c_policy,
    policy_type="nn_share", state_init=state_init, shocks=shocks,
)

print_elapsedtime(time.monotonic() - start_time)

## 3. The basis function $\mathcal{Q}(a)$

The generalized moment is

$$Q_t = \frac{1}{N}\sum_{i=1}^{N}\mathcal{Q}\big(\tilde{a}^i_t\big),
\qquad \tilde{a} = \frac{a - \mu_a}{\sigma_a},$$

so `gm_model.basis_fn` is $\mathcal{Q}$ applied agent by agent, and averaging over the agent
axis gives $Q_t$. Note the basis takes the **normalised** asset as its argument, which is
why `normalize_data(..., key="agt_s")` appears before the call.

$\mathcal{Q}$ is only identified up to an affine rescaling: the value network that consumes
$Q_t$ can undo any constant the basis is multiplied by, so the raw units of
$\mathcal{Q}$ carry no meaning. To make the picture readable we rescale so that the *average*
of $\mathcal{Q}$ equals average capital — if $\mathcal{Q}$ were the identity, the rescaled
curve would lie exactly on the dashed line.

That dashed line is the **chord** between the two endpoints of the curve, not a 45° line
(the vertical axis spans about 0.02 while the horizontal axis spans 40). It is there to
read off *concavity*: the curve lying above its own chord is the whole content of the panel.

In [ ]:
# Each value net carries its own basis. The two panels of Figure 3 in the paper happen to
# display different nets, so name the choice rather than leaving it implicit.
GM_NET_BASIS = 0   # whose basis Q_i(a) is drawn in this section  (Figure 3a)
GM_NET_VALUE = 2   # whose GM range is swept in the next section  (Figure 3b)

assert n_gm > 0, "This section needs a model trained with n_gm > 0 (e.g. game_nn_n50_1gm3)."


def gm_from_knormed(knormed, gm_model):
    """knormed (paths, n_agt, T) -> basis (paths, n_agt, T, n_gm), gm (paths, 1, T, n_gm)."""
    x = knormed.transpose(0, 2, 1)[:, :, :, None]      # (paths, T, n_agt, 1)
    basis = gm_model.basis_fn(x).numpy()               # (paths, T, n_agt, n_gm)
    basis = basis.transpose(0, 2, 1, 3)                # (paths, n_agt, T, n_gm)
    gm = np.mean(basis, axis=1, keepdims=True)         # (paths, 1,     T, n_gm)
    return basis, gm


k_cross = simul_data_nn["k_cross"]
knormed = ptrainer.init_ds.normalize_data(k_cross, key="agt_s")

basis_list = []
for vtr in vtrainers:
    b_i, _ = gm_from_knormed(knormed, vtr.gm_model)
    basis_list.append(b_i)

In [ ]:
b = basis_list[GM_NET_BASIS][..., 0].reshape(-1)
b = b / b.mean() * k_cross.mean()          # into units of capital (see the note above)
k = k_cross.reshape(-1)

mask = (k > 20) & (k < 60)                 # the bulk of the ergodic cross-section
k1, b1 = k[mask], b[mask]
order = np.argsort(k1)
k1, b1 = k1[order], b1[order]

fig = plt.figure(figsize=(7, 5.5))
ax = fig.add_subplot(111)
ax.plot([k1[0], k1[-1]], [b1[0], b1[-1]], "k--", label="chord")
ax.plot(k1, b1, linewidth=3, label=fr"$\mathcal{{Q}}_{GM_NET_BASIS}(a)$, rescaled")
ax.set_xlabel("individual asset $a$")
ax.set_ylabel(r"$\mathcal{Q}(a)$")
ax.legend()
plt.tight_layout()
if SAVE_FIGURES:
    os.makedirs("../output", exist_ok=True)
    fig.savefig("../output/Figure3a_KS_gm_basis_vs_asset.pdf")
plt.show()

print(f"curve spans {b1.min():.4f} to {b1.max():.4f}; the chord is the straight reference")

**Reading the panel.** $\mathcal{Q}$ is *concave* in individual assets. So a unit of assets
handed to a poor household raises the generalized moment by more than the same unit handed
to a rich one — the moment is not the mean, and the distribution enters the solution in a
way the Krusell–Smith approximation cannot express. That single fact is what drives the
redistribution result below.

## 4. From the generalized moment to the value function

Now hold $(a^i, Z, z^i)$ fixed at reference values and sweep the generalized moment across
its simulated range, to isolate $\partial V/\partial Q$. As in the previous section, the
moment is rescaled into units of capital, using `GM_NET_VALUE` for both the sweep range and
the scale.

In [ ]:
def value_fn_nn(basic_s, fm_extra, gm):
    """Evaluate every value net on a (paths, n_agt, T, .) panel of states.

    Returns (mean value, [value of each net]), all unnormalised and shaped
    (paths, n_agt, T).
    """
    shape_ref = basic_s.shape
    basic_s = ptrainer.init_ds.normalize_data(basic_s, key="basic_s")
    if ptrainer.init_ds.config["n_fm"] == 0:
        # drop aggregate capital: with no fixed moment it is not part of the state
        basic_s = np.concatenate([basic_s[..., 0:1], basic_s[..., 2:]], axis=-1)

    if fm_extra is not None:
        n_state = basic_s.shape[-1] + fm_extra.shape[-1]
        state_fix = np.concatenate([basic_s, fm_extra], axis=-1)
    else:
        n_state = basic_s.shape[-1]
        state_fix = basic_s

    if gm is not None:
        n_state += gm[0].shape[-1]
        state = []
        for i in range(len(vtrainers)):
            s = np.concatenate([state_fix, gm[i]], axis=-1)
            state.append(s.transpose((0, 2, 1, 3)).reshape((-1, n_agt, n_state)))
    else:
        s = state_fix.transpose((0, 2, 1, 3)).reshape((-1, n_agt, n_state))
        state = [s] * len(vtrainers)

    def to_panel(v_flat):
        v = ptrainer.init_ds.unnormalize_data(v_flat, key="value")
        v = v.reshape([shape_ref[0], shape_ref[2], shape_ref[1]])
        return np.transpose(v, (0, 2, 1))

    v_raw = [vtr.model(state[i]).numpy() for i, vtr in enumerate(vtrainers)]
    v_each = [to_panel(v) for v in v_raw]
    v_mean = to_panel(sum(v_raw) / len(v_raw))
    return v_mean, v_each


def gm_now_from_simulation(simul_data, shocks, nt=100, seed=None):
    """Sample `nt` dates from a simulation and return the GM of each value net there."""
    if seed is not None:
        np.random.seed(seed)
    k_cross_, csmp = simul_data["k_cross"], simul_data["csmp"]
    nt = min(nt, csmp.shape[-1])
    t_idx = np.random.choice(csmp.shape[-1], nt)
    knormed_now = ptrainer.init_ds.normalize_data(k_cross_[:, :, t_idx], key="agt_s")

    gm_now = []
    for vtr in vtrainers:
        _, g = gm_from_knormed(knormed_now, vtr.gm_model)
        gm_now.append(np.repeat(g, n_agt, axis=1))

    fm_extra_now = None
    if ptrainer.init_ds.config["n_fm"] == 2:
        knormed_mean = np.mean(knormed_now, axis=-2, keepdims=True)
        var = np.mean(knormed_now ** 2, axis=1, keepdims=True) - knormed_mean ** 2
        fm_extra_now = np.repeat(var, n_agt, axis=1)[:, :, :, None]
    return gm_now, fm_extra_now

In [ ]:
V_NET_PLOT = 0     # which value net's V is drawn; the paper's Figure 3(b) uses net 0
T_GM = 50          # number of GM draws per path

gm_now, fm_extra_now = gm_now_from_simulation(simul_data_nn, shocks, nt=100, seed=1)

# Sweep the generalized moment across the range value net GM_NET_VALUE actually visits.
gm_lo = gm_now[GM_NET_VALUE].min()
gm_hi = gm_now[GM_NET_VALUE].max()
gm_draw = np.random.uniform(gm_lo, gm_hi, (n_path, 1, T_GM, 1))
gm_draw = np.tile(gm_draw, (1, n_agt, 1, 1))
gm_in = [gm_draw.copy() for _ in vtrainers]

# Hold the individual state fixed: assets at the stationary-equilibrium level, and each of
# the four combinations of (employed / unemployed) x (low / high aggregate productivity).
base = np.zeros((n_path, n_agt, T_GM, 4))
base[..., 0] = mparam.k_ss                 # own assets
base[..., 1] = 0.0                         # aggregate capital: dropped below, n_fm == 0
Z_LOW, Z_HIGH = 0.99, 1.01

panels = {}
for Z, Z_label in [(Z_LOW, "Z^l"), (Z_HIGH, "Z^h")]:          # paper's panel order
    for z_i, z_label in [(0, "z^i_t = 0"), (1, "z^i_t = 1")]:
        s = base.copy()
        s[..., 2] = Z
        s[..., 3] = z_i
        _, v_each = value_fn_nn(s, fm_extra_now, gm_in)
        panels[(z_label, Z_label)] = v_each[V_NET_PLOT]

# The generalized moment in units of capital -- scaled by the SAME net whose range was swept.
gm_axis = gm_draw[:, 0:1, :, 0] / basis_list[GM_NET_VALUE].mean() * k_cross.mean()

assert 20 < gm_axis.mean() < 60, (
    f"The GM axis averages {gm_axis.mean():.2f}, which is not in units of capital "
    f"(K = {k_cross.mean():.1f}). Use GM_NET_VALUE for both the sweep range and the "
    f"axis scale."
)
print(f"GM axis spans [{gm_axis.min():.2f}, {gm_axis.max():.2f}], K = {k_cross.mean():.2f}")

In [ ]:
matplotlib.rcParams.update({"font.size": 13})
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, ((z_label, Z_label), v) in zip(axes.flat, panels.items()):
    ax.scatter(gm_axis.reshape(-1), v[:, 0:1, :].reshape(-1), s=4)
    ax.set_xlabel("generalized moment (units of capital)")
    ax.set_ylabel(f"$V$ (value net {V_NET_PLOT})")
    ax.set_title(rf"Value for ${z_label},\ Z_t = {Z_label}$")
plt.tight_layout()
if SAVE_FIGURES:
    os.makedirs("../output", exist_ok=True)
    fig.savefig("../output/Figure3b_KS_0fm1gm_gm_to_value.pdf", bbox_inches="tight")
plt.show()

**Reading the panels.** The value function is close to *linear* in the generalized moment,
and it **slopes downward**: with own assets held at the stationary level, a household is
worse off when the moment is higher.

Combine that with the concavity of $\mathcal{Q}$ from Section 3 and Section 3.2.2 of the
paper follows. Take one unit of assets from the richest households and give it to the
poorest: because $\mathcal{Q}$ is concave, this *raises* the generalized moment; because $V$
is decreasing in the moment, the "middle" households who are not part of the redistribution
are worse off on impact. A purely redistributive policy therefore has aggregate welfare
consequences even in the plain Krusell–Smith economy — which the mean-only representation
of notebook 01 cannot see, since it holds aggregate capital fixed.

Try `V_NET_PLOT = GM_NET_VALUE` to plot the value net whose own moment is being swept,
rather than net 0 as in the paper.

## 5. Aggregate dynamics: DeepHAM against Krusell–Smith

In [ ]:
def path_stats(simul_data, label=""):
    k_mean = np.mean(simul_data["k_cross"], axis=1)
    discount = np.power(mparam.beta, np.arange(simul_data["csmp"].shape[-1]))
    util_sum = np.sum(np.log(simul_data["csmp"]) * discount, axis=-1)
    print(
        f"{label:>8s}: total utility {util_sum.mean():9.5f} | "
        f"mean k {simul_data['k_cross'].mean():8.5f} | "
        f"std k {simul_data['k_cross'].std():8.5f} | "
        f"max k {simul_data['k_cross'].max():9.5f} | "
        f"max K {k_mean.max():8.5f}"
    )


path_stats(simul_data_bchmk, "KS")
path_stats(simul_data_nn, "DeepHAM")

In [ ]:
t0 = T_SIM // 2      # discard the first half as burn-in
path = 1             # which simulated economy to draw

for key, ylabel, ylim, fname in [
    ("k_cross", "$K_t$", [35.5, 42.5], "Figure7a_KS_NN_sim_K.pdf"),
    ("csmp", "$C_t$", [2.6, 3.0], "Figure7b_KS_NN_sim_C.pdf"),
]:
    bchmk = np.mean(simul_data_bchmk[key][path, :, t0:], axis=0)
    nn = np.mean(simul_data_nn[key][path, :, t0:], axis=0)

    fig = plt.figure(figsize=(8, 5.5))
    ax = fig.add_subplot(111)
    ax.plot(range(len(bchmk)), bchmk, "-.", color="b", linewidth=3, label="KS method")
    ax.plot(range(len(nn)), nn, color="r", linewidth=3, label="DeepHAM")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("$t$")
    if RUN_MODE == "production":
        ax.set_ylim(ylim)      # the window used in the paper
    ax.legend()
    plt.tight_layout()
    if SAVE_FIGURES:
        os.makedirs("../output", exist_ok=True)
        fig.savefig(f"../output/{fname}")
    plt.show()

In [ ]:
# The two solutions face identical shocks, so their aggregate paths should track each other
# closely. This checks that -- and would catch a model loaded with mismatched statistics.
K_bchmk = simul_data_bchmk["k_cross"][:, :, t0:].mean()
K_nn = simul_data_nn["k_cross"][:, :, t0:].mean()
rel_gap = abs(K_nn - K_bchmk) / K_bchmk
print(f"mean K: KS {K_bchmk:.3f} vs DeepHAM {K_nn:.3f}  ({100 * rel_gap:.2f}% apart)")
assert np.isfinite(rel_gap) and rel_gap < 0.25, (
    "DeepHAM and the KS benchmark disagree on the capital stock by more than 25% -- "
    "check that MODEL_DIR points at a converged run."
)

## Summary

* Under identical shocks, DeepHAM tracks the Krusell–Smith aggregate path closely, using a
  distribution summary it was never told to use.
* The learned basis $\mathcal{Q}$ is concave in individual assets, so the generalized moment
  weights poor households more heavily than the mean does.
* The value function is decreasing and roughly linear in that moment.

## Takeaway

The generalized moment is not a better-fitting statistic of the wealth distribution — it is
a statistic chosen *for the decision problem*, by the same gradients that train the value
function. What it learned here is that the relevant feature of the distribution is not
average wealth but something that loads on the poor, which is exactly why a pure
redistribution has aggregate effects in a model where the mean-only approximation says it
should not.